In [ ]:
!pip install -q -U google-generativeai
!pip install -U google-generativeai
!pip install -U google-generativeai datasets
import os
import google.generativeai as genai
from google.colab import userdata
from datasets import load_dataset, Dataset
import pandas as pd
from tqdm.auto import tqdm
from time import sleep
import json
import os
import time
from google.generativeai import GenerationConfig
import re
import random
from google.colab import drive

In [ ]:
os.environ["GOOGLE_API_KEY"] = "" #insira a API KEY aqui

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [ ]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [ ]:
model_name = "models/gemini-2.5-flash"


In [ ]:
model = genai.GenerativeModel(model_name)
response = model.generate_content("Explique em uma frase o que é uma estrela.")
print(response.text)


Uma estrela é um corpo celeste massivo, composto principalmente por gás, que gera sua própria luz e calor através de reações de fusão nuclear em seu núcleo.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datasets import load_dataset

dataset = load_dataset("NESPED-GEN/CNPJ", split="test").to_pandas()

In [ ]:
template = """
<instruction>
You will be provided with a database schema. Your task is to generate a {hardness} question:
1. Generate a natural language question based on the schema.
2. Write an SQL query that answers the generated question.
Ensure your response follows this structure:
1. Question
2. SQL Query
</instruction>


<example>
<schema>
CREATE TABLE employees (
  employee_id INTEGER PRIMARY KEY,
  first_name TEXT NOT NULL,
  last_name TEXT NOT NULL,
  department_id INTEGER NOT NULL,
  FOREIGN KEY (department_id) REFERENCES departments(department_id)
);
/*
2 rows from employees table:
employee_id  first_name  last_name  department_id
1            John         Doe          2
2            Jane         Smith        1
*/

CREATE TABLE departments (
  department_id INTEGER PRIMARY KEY,
  department_name TEXT NOT NULL
);
/*
2 rows from departments table:
department_id  department_name
1               Sales
2               Engineering
*/
</schema>

<question>
What are the names of all employees in the 'Sales' department?
</question>

<sql>
SELECT first_name, last_name
FROM employees e
JOIN departments d ON e.department_id = d.department_id
WHERE d.department_name = 'Sales';
</sql>
</example>

<output_instruction>
<input>
<schema>
{schema}
</schema>
</input>
Now, based on the provided schema, generate:
1. A natural language question.
2. An SQL query that answers the question.

Ensure your response follows this structure:
<question>
[Your generated question here]
</question>

<sql>
[Your SQL query here]
</sql>
</output_instruction>
"""

In [ ]:
# for m in genai.list_models():
#   if 'generateContent' in m.supported_generation_methods:
#     print(m.name)

In [ ]:
dataset.head()

,db_id,question,query,hardness,schema_SQLDatabase,schema_linking
0,cnpjENsample,Find the names of all companies that have part...,"SELECT c.name, p.name, ci.name FROM company AS...",extra,"CREATE TABLE company_size (\n code INT,...","{\n 'company': ['basic_cnpj', 'name'],\n 'es..."
1,cnpjENsample,What are the names and capital amounts of comp...,"SELECT c.name, c.capital\nFROM company AS c\nJ...",extra,"CREATE TABLE company_size (\n code INT,...","{\n 'company': ['basic_cnpj', 'name', 'capita..."
2,cnpjENsample,What are the names and capital amounts of comp...,"SELECT c.name, c.capital FROM company AS c JOI...",medium,"CREATE TABLE company_size (\n code INT,...","{\n 'company': ['basic_cnpj', 'name', 'capita..."
3,cnpjENsample,List the names and capital of companies that c...,"SELECT c.name, c.capital FROM company AS c JOI...",medium,"CREATE TABLE company_size (\n code INT,...","{\n 'company': ['basic_cnpj', 'name', 'capita..."
4,cnpjENsample,List the names of all companies with a capital...,"SELECT c.name, ln.description FROM company AS ...",medium,"CREATE TABLE company_size (\n code INT,...","{\n 'company': ['basic_cnpj', 'name', 'capita..."


In [ ]:
row = dataset.iloc[0]
schema = row['schema_SQLDatabase']

In [ ]:
def extract_question_and_sql(response):
    question_match = re.search(r"<question>\s*(.*?)\s*</question>", response, re.DOTALL)
    sql_match = re.search(r"<sql>\s*(.*?)\s*</sql>", response, re.DOTALL)

    question = question_match.group(1).strip() if question_match else None
    sql = sql_match.group(1).strip() if sql_match else None

    return {"question": question, "sql": sql}

In [ ]:
def generate_questions(model, schema, number_of_questions, batch_size=10, output_path="output.json", ):
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            existing_data = json.load(f)
        start_index = len(existing_data)
    else:
        existing_data = []
        start_index = 0

    number_of_questions -= start_index
    start_time = time.time()
    df = pd.DataFrame()
    for index  in tqdm(range(number_of_questions)):
        elements = ['easy', 'medium', 'hard', 'extra hard']
        probabilities = [0.24, 0.43, 0.16, 0.17]
        hardness = random.choices(elements, weights=probabilities, k=1)[0]

        attepts = 0

        while True:
            try:

                inp=template.format(schema=schema, hardness=hardness)
                response = model.generate_content(inp)
                ans = extract_question_and_sql(response.text)
                print(ans)
                existing_data.append(ans)
                attepts = 0
                break
            except Exception as e:
                attepts += 1
                print(index, e)
                sleep(20)
                if attepts > 5:
                    break


        if (index + 1) % batch_size == 0:
            with open(output_path, "w") as f:
                json.dump(existing_data, f)
            end_time = time.time()
            elapsed_time = end_time - start_time

            sleep(max(65 - elapsed_time, 0))
            start_time = time.time()

    with open(output_path, "w") as f:
        json.dump(existing_data, f)

In [ ]:
drive.mount('/content/drive')

genai.configure(api_key="")
generation_config = GenerationConfig(temperature=2)
model = genai.GenerativeModel(
    model_name,
    generation_config=generation_config
)


output_path = "" #insira o caminho do drive aqui
os.makedirs(os.path.dirname(output_path), exist_ok=True)


generate_questions(
    model,
    schema=schema,
    number_of_questions=100,
    batch_size=2,
    output_path=output_path
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


  0%|          | 0/100 [00:00<?, ?it/s]

{'question': "Retrieve the primary CNAE descriptions, the total count of companies, and the average capital for all 'main' establishments that are not federally responsible by 'Union', have opted for 'MEI Taxation' after January 1, 2020, and **do not** have any partners whose type is 'LEGAL ENTITY'. Only include CNAE descriptions for which there are at least 5 such companies and their average capital exceeds 500,000. Order the results by the total company count in descending order.", 'sql': "SELECT\n    cn.name AS primary_cnae_description,\n    COUNT(c.basic_cnpj) AS total_companies,\n    AVG(c.capital) AS average_capital\nFROM\n    establishment AS e\nJOIN\n    company AS c ON e.basic_cnpj = c.basic_cnpj\nJOIN\n    cnae AS cn ON e.primary_cnae_code = cn.code\nJOIN\n    taxation AS t ON e.basic_cnpj = t.basic_cnpj\nWHERE\n    e.main_or_branch = 'MAIN'\n    AND c.responsible_federal_entity <> 'Union'\n    AND t.option_for_mei_taxation = 'Y'\n    AND t.mei_taxation_option_date > '2020-01

KeyboardInterrupt: 